# Guitar Chord Lab 2：混合音からギターを分離してコード候補を表示する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shogo-ishikawa/2026_3Q_M4Th4/blob/main/projects/guitar-chord-lab/notebooks/02_separate_and_recognise.ipynb)　[GitHubで見る](https://github.com/shogo-ishikawa/2026_3Q_M4Th4/blob/main/projects/guitar-chord-lab/notebooks/02_separate_and_recognise.ipynb)

ギターと伴奏が混ざった音（混合音）から、**音源分離**でギターの音を取り出し、Notebook 1 と同じ方法でコード候補を推定します。分離する前と後で、結果を聴き比べ・見比べます。

**流れ**：セットアップ → 入力 → 再生 → 分離 → 同じ設定でコード推定 → 保存 → 条件を変えて試す

- **音源分離**は「混ざった音から特定の楽器の音を取り出す」処理で、**コード認識**は「取り出した音のコードを推定する」処理です。分離がうまくいかないと、後のコード認識にも影響します。
- 分離には Demucs の `htdemucs_6s` を使います。drums、bass、other、vocals、**guitar**、piano の6つの音を出力するモデルです（Demucs公式のREADMEで確認）。4音源のモデルの other はギターではないので使いません。
- GPU があると速く分離できます。Colabでは「ランタイム」→「ランタイムのタイプを変更」で GPU を選べます。CPU でも動きます（この教材を確認した環境（CPU 4コア）では、20秒の音源の分離に約12〜21秒。Colab の CPU ランタイムでは、より長くかかる可能性があります）。
- 分離モデルの重み（約53 MB）は、最初の実行時に Hugging Face Hub から取得します。Demucs の README には、`htdemucs_6s` の重みは Meta の社内データで学習されており、利用条件は明確でない（非商用の CC BY-NC が当てはまる可能性が高い）と書かれています。授業・研究の範囲で使い、分離した音を配布しないでください。

## 1. セットアップ

新しいColabでは、講義リポジトリをHTTPSで取得し、`projects/guitar-chord-lab/` の作品だけをインストールします。

| 項目 | 場所 |
| --- | --- |
| リポジトリの取得先 | `/content/2026_3Q_M4Th4` |
| 作業ディレクトリ（相対パスの基準） | `/content/2026_3Q_M4Th4/projects/guitar-chord-lab` |
| 結果の保存先 | `outputs/02_separation/<SOURCE>/`（作業ディレクトリからの相対パス） |

- 表示されるコミットは、使ったコードの版です。結果と一緒に記録してください。
- `BRANCH`（ブランチ）や `COMMIT`（コミット）を変えて実行し直すと、その版にそろえます。コードの版が変わったときは、セッションの再起動を求める表示が出ます。
- Colabのファイルはランタイムを終了すると消えます。残したい結果はダウンロードするか、Googleドライブにコピーしてください。
- ローカルで使う場合は、作品フォルダで `pip install -e ".[separation]"` を実行してから、`notebooks/` フォルダでこのNotebookを開きます。


In [ ]:
# セットアップ（最初に実行します。BRANCH や COMMIT を変えたときも実行し直します）
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/shogo-ishikawa/2026_3Q_M4Th4.git"
BRANCH = "main"  # 作業ブランチの内容を試すときは、ブランチ名に書き換えます
COMMIT = ""      # 実験を再現するときは、記録したコミット（例: "1a2b3c4"）を入れます
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/2026_3Q_M4Th4")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    # 実行するたびに BRANCH（COMMIT があればそのコミット）の内容にそろえます
    git = ["git", "-C", str(REPO_DIR)]
    before = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True).stdout
    subprocess.run(git + ["fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(git + ["checkout", "-q", "--detach", COMMIT or "FETCH_HEAD"], check=True)
    after = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True).stdout
    if before != after and "guitar_chord_lab" in sys.modules:
        raise RuntimeError("コードの版が変わりました。「ランタイム」→「セッションを再起動」を選んでから、このセルから実行し直してください")
    PROJECT_DIR = REPO_DIR / "projects" / "guitar-chord-lab"
    # この作品だけをインストールします（[separation]）
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_DIR}[separation]"], check=True)
else:
    # ローカルでは、notebooks/ フォルダで開いたこのNotebookの1つ上を作品フォルダとします
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)  # 以降の相対パスは、作品フォルダ（projects/guitar-chord-lab）が基準です
sys.path.insert(0, str(PROJECT_DIR / "src"))  # インストール直後でも import できるようにします
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("作業ディレクトリ:", Path.cwd())
print("使用しているコミット:", commit or "（Gitの管理外。配布ZIPなら release_info.json を参照）")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Audio, display

import guitar_chord_lab as gcl
from guitar_chord_lab.audio import load_audio, make_clip, prepare_for_analysis, save_audio
from guitar_chord_lab.evaluation import evaluate_segments, reference_for_clip
from guitar_chord_lab.experiment import load_accompaniment, mix_at_level
from guitar_chord_lab.plots import plot_analysis, plot_comparison
from guitar_chord_lab.recognition import ChordSettings, recognise_chords
from guitar_chord_lab.results import print_segments, save_analysis, save_json
from guitar_chord_lab.separation import SeparationError, describe_separation, separate_guitar

OUTPUT_DIR = Path("outputs/02_separation")


def play(clip):
    """音を再生します。音量を自動でそろえないので、小さい音は小さく聞こえます（1.0を超える部分は切って再生）。"""
    display(Audio(np.clip(clip.samples, -1.0, 1.0), rate=clip.sample_rate, normalize=False))


print("Guitar Chord Lab", gcl.__version__)
print("PyTorch", torch.__version__, "/ GPU（CUDA）が使えるか:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 入力する音源を選ぶ

| SOURCE | 内容 | 正解注釈 |
| --- | --- | --- |
| `"guitarset_mix"`（既定） | 実際のギター録音（GuitarSet）に、プログラムで作ったドラムを混ぜた音 | あり |
| `"demo_mix"` | 合成デモのギター風の音に、合成ドラムを混ぜた音 | あり |
| `"upload"` | 自分の合奏音源（WAV・FLAC・OGG・MP3） | なし |

伴奏の音量 `ACCOMPANIMENT_DB` は「伴奏のRMS − ギターのRMS」[dB] です（RMSは音量の目安で、無音を含む切り出し全体で計算します）。0 dB で同じくらい、+6 dB で伴奏のほうが約2倍の大きさです。

**注意**：合成デモのギター風の音は、Demucs にギターとして扱われにくいことを確認しています（分離した guitar がほぼ無音になる）。分離の確認には実際のギター録音を使ってください。

In [ ]:
SOURCE = "guitarset_mix"  # "guitarset_mix"、"demo_mix"、"upload"
ACCOMPANIMENT_DB = 6.0    # 伴奏の音量（伴奏のRMS − ギターのRMS）[dB]
START_S = 0.0             # 切り出しを始める時刻 [秒]
DURATION_S = 20.0         # 切り出す長さ [秒]
MIX_RATE = 44100          # 混合と分離に使うサンプリング周波数 [Hz]（Demucs と同じ）

guitar_clip = None        # 正解の単独ギター（自分の合奏音源では None）
ANNOTATION_PATH = None
if SOURCE in ["guitarset_mix", "demo_mix"]:
    from guitar_chord_lab.synth import normalise_peak, synthesise_drums, write_demo_dataset

    if SOURCE == "guitarset_mix":
        from guitar_chord_lab.guitarset import CITATION, EXAMPLE_TRACKS, prepare_track

        GUITAR_PATH, ANNOTATION_PATH = prepare_track(EXAMPLE_TRACKS[0], "data/raw/guitarset")
        print("出典:", CITATION)
    else:
        write_demo_dataset("data/generated/demo")
        GUITAR_PATH = "data/generated/demo/demo_a_guitar.wav"
        ANNOTATION_PATH = "data/generated/demo/demo_a_annotation.csv"
    guitar_clip = prepare_for_analysis(load_audio(GUITAR_PATH, START_S, DURATION_S), MIX_RATE)
    drums = normalise_peak(synthesise_drums(guitar_clip.duration_s, MIX_RATE, tempo_bpm=120, seed=0), 0.5)
    mix = mix_at_level(guitar_clip.samples[0], drums, ACCOMPANIMENT_DB)
    mixture_clip = make_clip(mix.mixture, MIX_RATE, "", guitar_clip.offset_s)
    print(f"伴奏の音量: 指定 {ACCOMPANIMENT_DB:+.1f} dB、測定 {mix.measured_db:+.2f} dB、音割れ防止の倍率 {mix.clip_guard_gain:.3f}")
elif SOURCE == "upload":
    from google.colab import files  # Colab でだけ使えます

    uploaded = files.upload()
    mixture_clip = load_audio(list(uploaded)[0], START_S, DURATION_S)
print(f"混合音: {mixture_clip.duration_s:.2f} 秒、{mixture_clip.sample_rate} Hz、{mixture_clip.channels} チャンネル")

## 3. 聴いてみる

混合音と、（あれば）元の単独ギターを聴き比べます。再生の音量は自動でそろえないので、音の大きさの違いもそのまま聞こえます。

In [ ]:
print("混合音")
play(mixture_clip)
if guitar_clip is not None:
    print("元の単独ギター（比較の基準）")
    play(guitar_clip)

## 4. ギターを分離する

`separate_guitar` は、モデルの取得、GPU/CPUの選択、分離を行います。同じ入力・同じ設定の結果は `outputs/cache/separation/` に保存し、次からは再利用します。

うまくいかないときは、原因と対処が表示されます（例：インターネットに接続できずモデルを取得できない、メモリが足りない）。

初回はモデルの重みを取得します。「HF_TOKEN を設定すると…」という Hugging Face の警告が表示されることがありますが、トークンがなくても取得できます。

In [ ]:
try:
    result = separate_guitar(mixture_clip, device="auto", cache_dir="outputs/cache/separation")
except SeparationError as error:
    print(error)
    raise

print(describe_separation(result))
print("分離したギター")
play(result.guitar)
print("ギター以外（混合音 − 分離したギター）")
play(result.rest)

## 5. 同じ設定でコード候補を推定する

混合音・分離したギター・（あれば）元の単独ギターを、**同じ認識器・同じ設定**で解析します。3つの音は同じ位置から切り出してあり、時刻の0秒がそろっています。

In [ ]:
settings = ChordSettings()
analyses = {}
if guitar_clip is not None:
    analyses["solo guitar"] = recognise_chords(guitar_clip, settings)
analyses["mixture"] = recognise_chords(mixture_clip, settings)
analyses["separated guitar"] = recognise_chords(result.guitar, settings)

reference = None  # 正解注釈（3つの音は同じ位置から切り出してあるので、単独ギターの切り出し位置で切り取ります）
if ANNOTATION_PATH:
    reference = reference_for_clip(ANNOTATION_PATH, guitar_clip)

if guitar_clip is not None:
    title = f"Accompaniment {ACCOMPANIMENT_DB:+.1f} dB"
else:
    title = "Uploaded recording (no added accompaniment)"
comparison = plot_comparison(analyses, reference, title)
plt.show()
if reference:
    for name, item in analyses.items():
        scores = evaluate_segments(reference, item.segments)
        print(f"{name:17} 正解率 {scores['accuracy']:.3f}  判定率 {scores['decided_ratio']:.3f}")

In [ ]:
print_segments(analyses["separated guitar"])
detail = plot_analysis(analyses["separated guitar"], "Separated guitar", reference)
plt.show()

## 6. 結果を保存する

混合音・分離したギター・ギター以外の音（WAV）、条件ごとの推定結果（CSV・JSON・図）、比較の図、混合と分離の条件（`conditions.json`）を、入力の種類（`SOURCE`）ごとのフォルダ `outputs/02_separation/<SOURCE>/` に保存します。混合の条件（使ったギターの録音、切り出し位置、伴奏、指定した音量と測った音量、音割れ防止の倍率）は、条件ごとの `analysis.json` にも記録します。

In [ ]:
RUN_DIR = OUTPUT_DIR / SOURCE  # 入力の種類ごとに分けて、別の入力の結果と混ざらないようにします
if guitar_clip is not None:  # 単独ギターに伴奏を混ぜた場合
    mixing = {
        "guitar": str(GUITAR_PATH), "annotation": str(ANNOTATION_PATH),
        "start_s": START_S, "duration_s": round(guitar_clip.duration_s, 6), "mix_sample_rate": MIX_RATE,
        "accompaniment": "synthetic drums (synthesise_drums, 120 BPM, seed 0)",
        "accompaniment_db": ACCOMPANIMENT_DB, "measured_db": round(mix.measured_db, 3),
        "clip_guard_gain": round(mix.clip_guard_gain, 5),
    }
else:  # 自分の合奏音源（伴奏は足していない）
    mixing = {"input": mixture_clip.source_path, "start_s": START_S, "duration_s": round(mixture_clip.duration_s, 6),
              "accompaniment": None}
save_audio(RUN_DIR / "mixture.wav", mixture_clip)
save_audio(RUN_DIR / "separated_guitar.wav", result.guitar)
save_audio(RUN_DIR / "without_guitar.wav", result.rest)
for name, item in analyses.items():
    extra = {"condition": name, "mixing": mixing, "separation": describe_separation(result)}
    if reference:
        extra["evaluation"] = evaluate_segments(reference, item.segments)
    save_analysis(item, RUN_DIR / name.replace(" ", "_"), extra)
comparison.savefig(RUN_DIR / "comparison.png", dpi=150)
save_json(RUN_DIR / "conditions.json", {"source": SOURCE, "mixing": mixing, "separation": describe_separation(result)})
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path)

## 7. 条件を変えて試す（練習）

### 練習1：伴奏の音量を変える
`ACCOMPANIMENT_DB` を 0、6、12 に変えて、2章から5章までを実行し直しましょう。混合音と分離したギターの正解率は、それぞれどう変わるでしょうか。

### 練習2：聴き比べる
分離したギターと元の単独ギターを聴き比べ、分離したギターに残っている伴奏の音や、欠けている音を探しましょう。「ギター以外」の音にギターが残っていないかも聴いてみましょう。

### 練習3：GPU と CPU の時間を比べる（GPUランタイムのとき）
`separate_guitar(mixture_clip, device="cpu")` と `device="cuda"` で、表示される `seconds`（分離にかかった時間）を比べましょう。キャッシュから読み込んだときは `from_cache` が True になり、`seconds` と `device` は最初に分離したときの値のままです。比べるときは `cache_dir` を指定しないでください。

### 練習4：自分の合奏音源で試す
`SOURCE = "upload"` にして、ギターを含む曲で試しましょう。正解注釈がないので、結果は耳で確かめます。

---

## 期待される観察と解答例

数値は素材や環境で変わります。ここでは既定の入力（GuitarSet の `01_Rock3-117-Bb_comp` の先頭20秒＋合成ドラム）で見られる傾向を説明します。

**練習1**
- 伴奏を大きくすると、混合音のクロマにドラムの雑音が広く混ざり、混合音の正解率は下がる傾向があります。
- 分離したギターは、伴奏の音量が変わっても結果が変わりにくい傾向があります。そのため、伴奏が大きいほど「分離後 − 分離前」の差が大きくなりやすいです。
- 伴奏が小さいときは差が小さくなります。Notebook 3 の5録音の実験では、伴奏 0 dB で分離後のほうが正解率の下がった録音もありました。
- 解答例：「分離が役立つかどうかは伴奏の大きさによって変わる。分離しても単独ギターの結果を超えるとは限らない」。

**練習2**
- ドラムのアタック（叩いた瞬間の音）が小さく残っていたり、ギターの高い音や余韻が弱くなっていたりします。分離は完全ではありません。

**練習3**
- GPU のほうが大幅に短い時間で終わります。CPU でも短い音源なら分離できます。

**練習4**
- 実際の合奏では、ギター以外の楽器（ベース、キーボード、歌）も和音の音を鳴らします。ギターが弾いているコードと曲全体の和声は一致しないことがある点に注意しましょう。

**合成デモ（`demo_mix`）について**：合成デモのギター風の音は、Demucs では主に piano や other に分けられ、guitar にはほとんど入りませんでした。分離後がほぼ無音で N ばかりになるのはこのためです。合成音の結果から分離の効果を判断しないでください。